# Lab 2 (Fixed): Feature Extraction from MSCOCO

In [1]:

import os
import numpy as np
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from pycocotools.coco import COCO
from PIL import Image
from transformers import BertTokenizer, BertModel

/home/BTECH_7TH_SEM/Desktop/MML-RL-and-NLP/MML/mml-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Paths (update as needed)
data_dir = "/home/BTECH_7TH_SEM/MS-COCO/val2017"
ann_file = "/home/BTECH_7TH_SEM/MS-COCO/annotations_trainval2017/annotations/captions_val2017.json"

# COCO dataset API
coco = COCO(ann_file)
img_ids = coco.getImgIds()[:25000]  # first 25k images
print("Total images:", len(img_ids))

# ---------------- Image Feature Extraction ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet = models.resnet50(pretrained=True)
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])  # remove last FC layer
resnet.eval().to(device)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def extract_image_feature(img_path):
    image = Image.open(img_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = resnet(image).squeeze().cpu().numpy()
    return feat

# ---------------- Caption Feature Extraction ----------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased").to(device)
bert.eval()

def extract_caption_feature(captions):
    embeddings = []
    for cap in captions:
        inputs = tokenizer(cap, return_tensors="pt", truncation=True, padding=True).to(device)
        with torch.no_grad():
            outputs = bert(**inputs)
            cls_emb = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
            embeddings.append(cls_emb)
    return np.mean(embeddings, axis=0)  # average 5 captions

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Total images: 5000


/home/BTECH_7TH_SEM/Desktop/MML-RL-and-NLP/MML/mml-venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/BTECH_7TH_SEM/Desktop/MML-RL-and-NLP/MML/mml-venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [3]:
# ---------------- Main Extraction ----------------
image_features = []
caption_features = []
labels = []  # placeholder: you should map each image to 80-class one-hot labels

for img_id in img_ids:
    img_info = coco.loadImgs(img_id)[0]
    img_path = os.path.join(data_dir, img_info['file_name'])
    
    # image features
    img_feat = extract_image_feature(img_path)
    image_features.append(img_feat)
    
    # captions
    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns = coco.loadAnns(ann_ids)
    caps = [ann['caption'] for ann in anns[:5]]  # take first 5 captions
    cap_feat = extract_caption_feature(caps)
    caption_features.append(cap_feat)
    
    # labels (dummy, replace with your mapping)
    labels.append(np.zeros(80))

image_features = np.array(image_features)
caption_features = np.array(caption_features)
labels = np.array(labels)

print("Image features:", image_features.shape)
print("Caption features:", caption_features.shape)
print("Labels:", labels.shape)# Paths (update as needed)

# COCO dataset API
coco = COCO(ann_file)
img_ids = coco.getImgIds()[:25000]  # first 25k images
print("Total images:", len(img_ids))

# ---------------- Image Feature Extraction ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet = models.resnet50(pretrained=True)
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])  # remove last FC layer
resnet.eval().to(device)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def extract_image_feature(img_path):
    image = Image.open(img_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = resnet(image).squeeze().cpu().numpy()
    return feat

# ---------------- Caption Feature Extraction ----------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased").to(device)
bert.eval()

def extract_caption_feature(captions):
    embeddings = []
    for cap in captions:
        inputs = tokenizer(cap, return_tensors="pt", truncation=True, padding=True).to(device)
        with torch.no_grad():
            outputs = bert(**inputs)
            cls_emb = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
            embeddings.append(cls_emb)
    return np.mean(embeddings, axis=0)  # average 5 captions

# ---------------- Main Extraction ----------------
image_features = []
caption_features = []
labels = []  # placeholder: you should map each image to 80-class one-hot labels

for img_id in img_ids:
    img_info = coco.loadImgs(img_id)[0]
    img_path = os.path.join(data_dir, img_info['file_name'])
    
    # image features
    img_feat = extract_image_feature(img_path)
    image_features.append(img_feat)
    
    # captions
    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns = coco.loadAnns(ann_ids)
    caps = [ann['caption'] for ann in anns[:5]]  # take firs

# Save
np.save("coco_features/image_features.npy", image_features)
np.save("coco_features/caption_features.npy", caption_features)
np.save("coco_features/labels.npy", labels)
print("Features saved!")

Image features: (5000, 2048)
Caption features: (5000, 768)
Labels: (5000, 80)
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Total images: 5000
Features saved!
